# 05 - Pipeline Tracking & Manifest Generation

## Prerequisites
- `01_face_detection_extraction.ipynb` completed
- `02_spatial_feature_extraction.ipynb` completed
- `03_frequency_feature_extraction.ipynb` completed
- `04_semantic_feature_extraction.ipynb` completed

All 4 notebooks above must be finished before running this one.

## Outputs
- `pipeline_tracking.csv` — Per-dataset counts at each pipeline stage
- `preprocessing_manifest.csv` — Sample manifest for `train_sections5_end.ipynb`

In [ ]:
import os
import time
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

print("Imports ready")

In [ ]:
class Config:
    BASE_DIR = Path(".")
    PREPROCESSED_DIR = BASE_DIR / "preprocessed_faces_integrated"
    FEATURES_DIR = BASE_DIR / "extracted_features_integrated"
    INTEGRATED_METADATA_PATH = BASE_DIR / "integrated_nocuda.csv"

config = Config()
print("Config ready")

In [ ]:
# ============================================================================
# LOAD METADATA FROM NOTEBOOK 01
# ============================================================================

meta_path = config.INTEGRATED_METADATA_PATH
if meta_path.exists():
    integrated_metadata = pd.read_csv(meta_path)
    print(f"Loaded metadata: {len(integrated_metadata)} rows from {meta_path}")
    print(f"Datasets: {integrated_metadata['dataset'].unique().tolist()}")
else:
    print(f"WARNING: {meta_path} not found!")
    print("Run 01_face_detection_extraction.ipynb first.")
    integrated_metadata = None

## Pipeline Tracking CSV

In [ ]:
# ============================================================================
# PIPELINE TRACKING CSV - Counts per dataset at each stage
# ============================================================================

def generate_pipeline_tracking_csv(metadata_df, preprocessed_dir, features_dir, save_path=None):
    """
    Generate a CSV that tracks how many items from each dataset 
    go through each pipeline stage: loading, preprocessing, feature extraction, training.
    
    Shows counts broken down by dataset, label, and media type.
    """
    import time  # DEBUG: for timing each stage
    
    save_path = save_path or config.BASE_DIR / 'pipeline_tracking.csv'
    
    prefix_to_dataset = {
        'FFPP_': 'FaceForensics++',
        'CelebDF_': 'Celeb-DF',
        'DFDC_': 'DFDC',
        'HiDF_': 'HiDF'
    }
    
    tracking_rows = []
    
    # ========== DEBUG: Stage 1 Start ==========
    print("[DEBUG] >>> STAGE 1: Loaded metadata - START")
    stage1_start = time.time()
    
    # --- Stage 1: Loaded metadata ---
    if metadata_df is not None and len(metadata_df) > 0:
        print(f"[DEBUG] Stage 1: metadata has {len(metadata_df)} rows, {metadata_df['dataset'].nunique()} datasets")
        for ds in metadata_df['dataset'].unique():
            ds_df = metadata_df[metadata_df['dataset'] == ds]
            real = len(ds_df[ds_df['label'] == 'real'])
            fake = len(ds_df[ds_df['label'] == 'fake'])
            
            # Count videos vs images - VECTORIZED instead of iterrows (was slow!)
            extensions = ds_df['path'].apply(lambda p: Path(p).suffix.lower())
            img_mask = extensions.isin(['.jpg', '.jpeg', '.png', '.bmp'])
            img_count = int(img_mask.sum())
            vid_count = len(ds_df) - img_count
            
            print(f"[DEBUG]   Dataset '{ds}': total={len(ds_df)}, real={real}, fake={fake}, vid={vid_count}, img={img_count}")
            
            tracking_rows.append({
                'Stage': '1_Loaded',
                'Dataset': ds,
                'Total': len(ds_df),
                'Real': real,
                'Fake': fake,
                'Videos': vid_count,
                'Images': img_count
            })
    else:
        print("[DEBUG] Stage 1: metadata_df is None or empty!")
    
    stage1_time = time.time() - stage1_start
    print(f"[DEBUG] <<< STAGE 1 COMPLETE in {stage1_time:.2f}s, tracking_rows so far: {len(tracking_rows)}")
    
    # ========== DEBUG: Stage 2 Start ==========
    print(f"\n[DEBUG] >>> STAGE 2: Preprocessed faces - START")
    print(f"[DEBUG] Stage 2: preprocessed_dir = {preprocessed_dir}")
    stage2_start = time.time()
    stage2_count = 0
    
    # --- Stage 2: Preprocessed faces ---
    for label in ['real', 'fake']:
        label_dir = preprocessed_dir / label
        if not label_dir.exists():
            print(f"[DEBUG]   Stage 2: {label_dir} does NOT exist, skipping")
            continue
        
        # DEBUG: Count total dirs first so we can show progress
        all_dirs = [d for d in label_dir.iterdir() if d.is_dir()]
        total_dirs = len(all_dirs)
        print(f"[DEBUG]   Stage 2 '{label}': found {total_dirs} directories to scan")
        
        for i, video_dir in enumerate(all_dirs):
            # DEBUG: Progress every 500 directories
            if (i + 1) % 500 == 0 or i == 0:
                elapsed = time.time() - stage2_start
                print(f"[DEBUG]   Stage 2 '{label}': scanning dir {i+1}/{total_dirs} ({elapsed:.1f}s elapsed)")
            
            video_id = video_dir.name
            # OPTIMIZATION: Use os.listdir + filter instead of glob (faster on Windows)
            try:
                num_frames = sum(1 for f in os.listdir(video_dir) if f.endswith('.png'))
            except OSError as e:
                print(f"[DEBUG]   Stage 2 ERROR reading {video_dir}: {e}")
                continue
            if num_frames == 0:
                continue
            
            ds_name = None
            for prefix, name in prefix_to_dataset.items():
                if video_id.startswith(prefix):
                    ds_name = name
                    break
            if ds_name is None:
                ds_name = 'Unknown'
            
            tracking_rows.append({
                'Stage': '2_Preprocessed',
                'Dataset': ds_name,
                'Label': label,
                'Videos_or_Items': 1,
                'Face_Images': num_frames
            })
            stage2_count += 1
        
        print(f"[DEBUG]   Stage 2 '{label}': DONE scanning {total_dirs} dirs")
    
    stage2_time = time.time() - stage2_start
    print(f"[DEBUG] <<< STAGE 2 COMPLETE in {stage2_time:.2f}s, added {stage2_count} rows, tracking_rows total: {len(tracking_rows)}")
    
    # ========== DEBUG: Stage 3 Start ==========
    print(f"\n[DEBUG] >>> STAGE 3: Feature extraction - START")
    print(f"[DEBUG] Stage 3: features_dir = {features_dir}")
    stage3_start = time.time()
    stage3_count = 0
    
    # --- Stage 3: Feature extraction ---
    for domain in ['spatial', 'frequency', 'semantic']:
        domain_start = time.time()
        print(f"[DEBUG]   Stage 3 domain '{domain}': starting...")
        
        for label in ['real', 'fake']:
            domain_label_dir = features_dir / domain / label
            if not domain_label_dir.exists():
                print(f"[DEBUG]     {domain}/{label}: directory does NOT exist, skipping")
                continue
            
            # DEBUG: Count total dirs first 
            all_dirs = [d for d in domain_label_dir.iterdir() if d.is_dir()]
            total_dirs = len(all_dirs)
            print(f"[DEBUG]     {domain}/{label}: found {total_dirs} directories to scan")
            
            for i, video_dir in enumerate(all_dirs):
                # DEBUG: Progress every 1000 directories
                if (i + 1) % 1000 == 0 or i == 0:
                    elapsed = time.time() - domain_start
                    print(f"[DEBUG]     {domain}/{label}: scanning dir {i+1}/{total_dirs} ({elapsed:.1f}s elapsed)")
                
                video_id = video_dir.name
                # OPTIMIZATION: Use os.listdir + filter instead of glob (faster on Windows)
                try:
                    num_features = sum(1 for f in os.listdir(video_dir) if f.endswith('.npy'))
                except OSError as e:
                    print(f"[DEBUG]     Stage 3 ERROR reading {video_dir}: {e}")
                    continue
                if num_features == 0:
                    continue
                
                ds_name = None
                for prefix, name in prefix_to_dataset.items():
                    if video_id.startswith(prefix):
                        ds_name = name
                        break
                if ds_name is None:
                    ds_name = 'Unknown'
                
                tracking_rows.append({
                    'Stage': f'3_Features_{domain}',
                    'Dataset': ds_name,
                    'Label': label,
                    'Videos_or_Items': 1,
                    'Feature_Files': num_features
                })
                stage3_count += 1
            
            print(f"[DEBUG]     {domain}/{label}: DONE scanning {total_dirs} dirs")
        
        domain_time = time.time() - domain_start
        print(f"[DEBUG]   Stage 3 domain '{domain}': COMPLETE in {domain_time:.2f}s")
    
    stage3_time = time.time() - stage3_start
    print(f"[DEBUG] <<< STAGE 3 COMPLETE in {stage3_time:.2f}s, added {stage3_count} rows, tracking_rows total: {len(tracking_rows)}")
    
    # ========== DEBUG: Aggregation Start ==========
    print(f"\n[DEBUG] >>> AGGREGATION: Building summary from {len(tracking_rows)} raw rows")
    agg_start = time.time()
    
    # Build the final tracking dataframe
    raw_df = pd.DataFrame(tracking_rows)
    print(f"[DEBUG] raw_df shape: {raw_df.shape}, columns: {list(raw_df.columns)}")
    
    # Aggregate into a clean summary
    summary_rows = []
    
    # Stage 1 summary (already aggregated)
    stage1 = raw_df[raw_df['Stage'] == '1_Loaded']
    print(f"[DEBUG] Stage 1 rows in raw_df: {len(stage1)}")
    for _, row in stage1.iterrows():
        summary_rows.append(row.to_dict())
    
    # Stage 2 summary (aggregate)
    stage2 = raw_df[raw_df['Stage'] == '2_Preprocessed']
    print(f"[DEBUG] Stage 2 rows in raw_df: {len(stage2)}")
    if len(stage2) > 0:
        for ds in stage2['Dataset'].unique():
            ds_data = stage2[stage2['Dataset'] == ds]
            for label in ['real', 'fake']:
                label_data = ds_data[ds_data['Label'] == label]
                if len(label_data) > 0:
                    summary_rows.append({
                        'Stage': '2_Preprocessed',
                        'Dataset': ds,
                        'Label': label,
                        'Items_Count': int(label_data['Videos_or_Items'].sum()),
                        'Total_Face_Images': int(label_data['Face_Images'].sum())
                    })
    
    # Stage 3 summary (aggregate per domain)
    for domain in ['spatial', 'frequency', 'semantic']:
        stage3 = raw_df[raw_df['Stage'] == f'3_Features_{domain}']
        print(f"[DEBUG] Stage 3 '{domain}' rows in raw_df: {len(stage3)}")
        if len(stage3) > 0:
            for ds in stage3['Dataset'].unique():
                ds_data = stage3[stage3['Dataset'] == ds]
                for label in ['real', 'fake']:
                    label_data = ds_data[ds_data['Label'] == label]
                    if len(label_data) > 0:
                        summary_rows.append({
                            'Stage': f'3_Features_{domain}',
                            'Dataset': ds,
                            'Label': label,
                            'Items_Count': int(label_data['Videos_or_Items'].sum()),
                            'Total_Feature_Files': int(label_data['Feature_Files'].sum())
                        })
    
    agg_time = time.time() - agg_start
    print(f"[DEBUG] <<< AGGREGATION COMPLETE in {agg_time:.2f}s, summary has {len(summary_rows)} rows")
    
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(save_path, index=False)
    print(f"[DEBUG] CSV saved to {save_path}")
    
    # Print a clean overview
    print(f"\n{'='*70}")
    print("PIPELINE TRACKING SUMMARY")
    print(f"{'='*70}")
    
    # Stage 1
    print("\n--- Stage 1: Loaded Metadata ---")
    s1 = summary_df[summary_df['Stage'] == '1_Loaded']
    if len(s1) > 0:
        for _, r in s1.iterrows():
            print(f"  {r['Dataset']:20s}: Total={int(r.get('Total',0)):5d}, Real={int(r.get('Real',0)):5d}, Fake={int(r.get('Fake',0)):5d}, Videos={int(r.get('Videos',0)):5d}, Images={int(r.get('Images',0)):5d}")
    
    # Stage 2
    print("\n--- Stage 2: Preprocessed Faces ---")
    s2 = summary_df[summary_df['Stage'] == '2_Preprocessed']
    if len(s2) > 0:
        for ds in s2['Dataset'].unique():
            ds_data = s2[s2['Dataset'] == ds]
            items = int(ds_data['Items_Count'].sum())
            faces = int(ds_data['Total_Face_Images'].sum())
            real_items = int(ds_data[ds_data['Label']=='real']['Items_Count'].sum()) if 'real' in ds_data['Label'].values else 0
            fake_items = int(ds_data[ds_data['Label']=='fake']['Items_Count'].sum()) if 'fake' in ds_data['Label'].values else 0
            print(f"  {ds:20s}: Items={items:5d} (Real={real_items}, Fake={fake_items}), Total_Faces={faces:6d}")
    
    # Stage 3
    print("\n--- Stage 3: Feature Extraction ---")
    for domain in ['spatial', 'frequency', 'semantic']:
        s3 = summary_df[summary_df['Stage'] == f'3_Features_{domain}']
        if len(s3) > 0:
            total_features = int(s3['Total_Feature_Files'].sum()) if 'Total_Feature_Files' in s3.columns else 0
            print(f"  {domain.upper():12s}: Total={total_features:6d}", end="")
            for ds in s3['Dataset'].unique():
                ds_feats = int(s3[s3['Dataset']==ds]['Total_Feature_Files'].sum())
                print(f"  | {ds}={ds_feats}", end="")
            print()
    
    print(f"\nTracking CSV saved to: {save_path}")
    total_time = stage1_time + stage2_time + stage3_time + agg_time
    print(f"[DEBUG] TOTAL pipeline tracking time: {total_time:.2f}s")
    return summary_df

# Generate tracking CSV
print("[DEBUG] Calling generate_pipeline_tracking_csv...")
tracking_df = generate_pipeline_tracking_csv(
    integrated_metadata, 
    config.PREPROCESSED_DIR, 
    config.FEATURES_DIR
)
print("[DEBUG] generate_pipeline_tracking_csv FINISHED")

## Preprocessing Manifest

In [ ]:
# ============================================================================
# SAVE PREPROCESSING MANIFEST
# ============================================================================
# Scans the features directory and saves a CSV listing every sample that has
# complete features across all 3 domains (spatial, frequency, semantic).
# The training notebook loads this manifest instead of re-scanning the filesystem.

def save_preprocessing_manifest(features_dir, save_path):
    """Scan features directory and save manifest of complete samples."""
    prefix_to_dataset = {
        'FFPP_': 'FaceForensics++',
        'CelebDF_': 'Celeb-DF',
        'DFDC_': 'DFDC',
        'HiDF_': 'HiDF'
    }
    required_domains = ['spatial', 'frequency', 'semantic']

    rows = []
    for label in ['real', 'fake']:
        spatial_label_dir = features_dir / 'spatial' / label
        if not spatial_label_dir.exists():
            continue

        video_dirs = [d for d in spatial_label_dir.iterdir() if d.is_dir()]
        print(f"  Scanning {label}: {len(video_dirs)} video directories...")

        for video_dir in tqdm(video_dirs, desc=f"Building manifest ({label})"):
            video_id = video_dir.name

            # Determine dataset from prefix
            dataset_name = None
            for prefix, ds_name in prefix_to_dataset.items():
                if video_id.startswith(prefix):
                    dataset_name = ds_name
                    break
            if dataset_name is None:
                continue

            # Check each frame for complete features across all domains
            for feature_file in video_dir.glob('*.npy'):
                frame_name = feature_file.stem
                all_exist = True
                for domain in required_domains:
                    path = features_dir / domain / label / video_id / f'{frame_name}.npy'
                    if not path.exists():
                        all_exist = False
                        break
                if all_exist:
                    rows.append({
                        'video_id': video_id,
                        'label': label,
                        'dataset': dataset_name,
                        'frame_name': frame_name
                    })

    manifest_df = pd.DataFrame(rows)
    manifest_df.to_csv(save_path, index=False)

    print(f"\n{'='*70}")
    print("PREPROCESSING MANIFEST SAVED")
    print(f"{'='*70}")
    print(f"Total samples with complete features: {len(manifest_df)}")
    if len(manifest_df) > 0:
        print(f"\nBy label:")
        for lbl, count in manifest_df['label'].value_counts().items():
            print(f"  {lbl}: {count}")
        print(f"\nBy dataset:")
        for ds, count in manifest_df['dataset'].value_counts().items():
            print(f"  {ds}: {count}")
    print(f"\nSaved to: {save_path}")
    print(f"\nNext step: Open train_sections5_end.ipynb to train the model.")
    return manifest_df

manifest_path = config.BASE_DIR / 'preprocessing_manifest.csv'
manifest_df = save_preprocessing_manifest(config.FEATURES_DIR, manifest_path)

## Final Validation

In [ ]:
# ============================================================================
# FINAL VALIDATION
# ============================================================================

print(f"\n{'='*70}")
print("FINAL VALIDATION")
print(f"{'='*70}")

# Check output files exist
for fname in ["pipeline_tracking.csv", "preprocessing_manifest.csv"]:
    fpath = config.BASE_DIR / fname
    if fpath.exists():
        df = pd.read_csv(fpath)
        print(f"  {fname}: {len(df)} rows")
    else:
        print(f"  {fname}: NOT FOUND!")

# Show manifest summary
manifest_path = config.BASE_DIR / "preprocessing_manifest.csv"
if manifest_path.exists():
    mf = pd.read_csv(manifest_path)
    print(f"\nManifest summary:")
    print(f"  Total samples with complete features: {len(mf)}")
    if len(mf) > 0:
        for lbl, cnt in mf["label"].value_counts().items():
            print(f"    {lbl}: {cnt}")
        for ds, cnt in mf["dataset"].value_counts().items():
            print(f"    {ds}: {cnt}")
    print(f"\nPreprocessing pipeline complete!")
    print("Next step: Open train_sections5_end.ipynb to train the model.")
else:
    print(f"\nManifest not generated. Check errors above.")